# Softmax Regression — GLM / Exponential Family Algorithm

## Learning Objectives
- Write the **categorical distribution** in exponential family form $p(y \mid \eta) = b(y)\exp(\eta^T T(y) - a(\eta))$
- Identify the **natural parameters** $\eta_k = \log(\theta_k / \theta_K)$ (log-odds vs reference class $K$)
- Derive the **canonical response function** $\mu_k = \partial a / \partial \eta_k$ and show it equals the softmax function
- Understand how the **GLM recipe** (random + systematic + link components) produces softmax regression
- Implement `fit_softmax_glm` using per-class IRLS (Newton's method), showing quadratic convergence vs gradient descent

## Problem Statement

Given training data $\{(x^{(i)}, y^{(i)})\}_{i=1}^{m}$ with $y^{(i)} \in \{1, \ldots, K\}$ (categorical response), the **GLM framework** asks:

> What distribution does $y$ follow? What is the canonical link? What does the MLE look like?

For binary classification, Bernoulli + logit link gives logistic regression. The natural extension to $K$ classes is **categorical + softmax link**, which is softmax regression.

---

### GLM Three-Component View

| Component | Binary ($K=2$) — Logistic | Multi-class ($K>2$) — Softmax |
|-----------|--------------------------|-------------------------------|
| **Random** | $y \sim \text{Bernoulli}(\mu)$ | $y \sim \text{Categorical}(\mu_1, \ldots, \mu_K)$ |
| **Systematic** | $\eta = \theta^T x$ (scalar) | $\eta_k = \theta_k^T x$, $k = 1, \ldots, K{-}1$ |
| **Canonical link** | Logit: $\eta = \log\frac{\mu}{1-\mu}$ | Log-odds: $\eta_k = \log\frac{\mu_k}{\mu_K}$ |
| **Inverse link** | Sigmoid: $\mu = \sigma(\eta)$ | Softmax: $\mu_k = e^{\eta_k} / \sum_j e^{\eta_j}$ |
| **Variance** $V(\mu)$ | $\mu(1-\mu)$ | $\mu_k(1-\mu_k)$ (marginal) |
| **Universal gradient** | $X^T(y - \mu)$ | $X^T(y_k - \mu_k)$ per class |

**Key observation**: Softmax regression is the **multi-class GLM with categorical distribution and log-odds canonical link**. The derivation from exponential family theory guarantees the softmax function as the unique canonical response function.

In [ ]:
from IPython.display import SVG, display

svg = """
<svg xmlns="http://www.w3.org/2000/svg" width="680" height="320" viewBox="0 0 680 320">
  <rect width="680" height="320" fill="#fafafa" rx="8"/>
  <text x="340" y="24" text-anchor="middle" font-size="13" font-weight="bold" fill="#222">Categorical Distribution &#x2192; Exponential Family &#x2192; Softmax (GLM Derivation)</text>

  <!-- Step 1 box -->
  <rect x="20" y="45" width="190" height="180" rx="6" fill="#e8f4fd" stroke="#4a90d9" stroke-width="2"/>
  <text x="115" y="68" text-anchor="middle" font-size="11" font-weight="bold" fill="#4a90d9">Step 1: Categorical</text>
  <line x1="30" y1="74" x2="200" y2="74" stroke="#4a90d9" stroke-width="1" stroke-dasharray="3,2"/>
  <text x="115" y="92" text-anchor="middle" font-size="9" fill="#333">p(y|&#x3B8;) = &#x220F; &#x3B8;&#x2C7C;^&#x1D30;(y=k)</text>
  <text x="115" y="110" text-anchor="middle" font-size="9" fill="#555">&#x3B8;&#x2C7C; = P(y=k), &#x2211;&#x3B8;&#x2C7C;=1</text>
  <text x="115" y="135" text-anchor="middle" font-size="9" fill="#333">Write as exp family:</text>
  <text x="115" y="152" text-anchor="middle" font-size="9" fill="#555">p(y|&#x3B7;) = exp(&#x3B7;&#x1D40;T(y) &#x2212; a(&#x3B7;))</text>
  <text x="115" y="172" text-anchor="middle" font-size="9" fill="#333">Natural parameter:</text>
  <text x="115" y="190" text-anchor="middle" font-size="9" fill="#555">&#x3B7;&#x2C7C; = log(&#x3B8;&#x2C7C;/&#x3B8;&#x1D37;), k=1..K-1</text>
  <text x="115" y="210" text-anchor="middle" font-size="9" fill="#333">Log-partition:</text>
  <text x="115" y="222" text-anchor="middle" font-size="9" fill="#555">a(&#x3B7;) = log(1 + &#x2211; e^&#x3B7;&#x2C7C;)</text>

  <!-- Arrow 1 -->
  <line x1="212" y1="135" x2="248" y2="135" stroke="#888" stroke-width="2"/>
  <polygon points="248,130 258,135 248,140" fill="#888"/>
  <text x="230" y="127" text-anchor="middle" font-size="8" fill="#888">MLE</text>

  <!-- Step 2 box -->
  <rect x="260" y="45" width="160" height="180" rx="6" fill="#fff8e8" stroke="#e07b00" stroke-width="2"/>
  <text x="340" y="68" text-anchor="middle" font-size="11" font-weight="bold" fill="#e07b00">Step 2: GLM Recipe</text>
  <line x1="270" y1="74" x2="410" y2="74" stroke="#e07b00" stroke-width="1" stroke-dasharray="3,2"/>
  <text x="340" y="92" text-anchor="middle" font-size="9" fill="#333">Set &#x3B7;&#x2C7C; = &#x3B8;&#x2C7C;&#x1D40;x (linear)</text>
  <text x="340" y="112" text-anchor="middle" font-size="9" fill="#555">&#x3B7; = [&#x3B8;&#x2081;&#x1D40;x, ..., &#x3B8;&#x1D37;&#x208B;&#x2081;&#x1D40;x]</text>
  <text x="340" y="140" text-anchor="middle" font-size="9" fill="#333">Canonical response:</text>
  <text x="340" y="158" text-anchor="middle" font-size="9" fill="#555">&#x3BC;&#x2C7C; = &#x2202;a/&#x2202;&#x3B7;&#x2C7C;</text>
  <text x="340" y="178" text-anchor="middle" font-size="9" fill="#555">= e^&#x3B7;&#x2C7C; / (1+&#x2211;e^&#x3B7;&#x2C7C;)</text>
  <text x="340" y="204" text-anchor="middle" font-size="9" fill="#333">Setting &#x3B7;&#x1D37;=0 (reference):</text>
  <text x="340" y="220" text-anchor="middle" font-size="9" fill="#555">&#x3BC;&#x2C7C; = e^&#x3B7;&#x2C7C; / &#x2211;e^&#x3B7;&#x2C7C; = softmax!</text>

  <!-- Arrow 2 -->
  <line x1="422" y1="135" x2="458" y2="135" stroke="#888" stroke-width="2"/>
  <polygon points="458,130 468,135 458,140" fill="#888"/>
  <text x="440" y="127" text-anchor="middle" font-size="8" fill="#888">&#x2207;&#x2113;</text>

  <!-- Step 3 box -->
  <rect x="470" y="45" width="190" height="180" rx="6" fill="#edfaed" stroke="#1a6e2e" stroke-width="2"/>
  <text x="565" y="68" text-anchor="middle" font-size="11" font-weight="bold" fill="#1a6e2e">Step 3: Gradient</text>
  <line x1="480" y1="74" x2="650" y2="74" stroke="#1a6e2e" stroke-width="1" stroke-dasharray="3,2"/>
  <text x="565" y="92" text-anchor="middle" font-size="9" fill="#333">For canonical link:</text>
  <text x="565" y="112" text-anchor="middle" font-size="9" fill="#555">&#x2207;&#x2113;&#x2C7C; = X&#x1D40;(y&#x2C7C; &#x2212; &#x3BC;&#x2C7C;)</text>
  <text x="565" y="135" text-anchor="middle" font-size="9" fill="#333">Hessian (per class k):</text>
  <text x="565" y="153" text-anchor="middle" font-size="9" fill="#555">H&#x2C7C;&#x2C7C; = &#x2212;X&#x1D40;W&#x2C7C;X</text>
  <text x="565" y="170" text-anchor="middle" font-size="9" fill="#555">W&#x2C7C; = diag(&#x3BC;&#x2C7C;(1&#x2212;&#x3BC;&#x2C7C;))</text>
  <text x="565" y="195" text-anchor="middle" font-size="9" fill="#333">IRLS Newton step:</text>
  <text x="565" y="213" text-anchor="middle" font-size="9" fill="#555">&#x3B8;&#x2C7C; &#x2190; &#x3B8;&#x2C7C; + (X&#x1D40;W&#x2C7C;X)&#x207B;&#xB9;X&#x1D40;(y&#x2C7C;&#x2212;&#x3BC;&#x2C7C;)</text>
  <text x="565" y="225" text-anchor="middle" font-size="9" font-weight="bold" fill="#1a6e2e">per class k = 1..K</text>

  <!-- Bottom row: comparison -->
  <rect x="20" y="245" width="640" height="55" rx="6" fill="#f5f5f5" stroke="#bbb" stroke-width="1"/>
  <text x="340" y="263" text-anchor="middle" font-size="10" font-weight="bold" fill="#333">Comparison: GLM (IRLS) vs Gradient Descent</text>
  <text x="175" y="282" text-anchor="middle" font-size="9" fill="#555">Gradient descent: &#x398; &#x2190; &#x398; &#x2212; &#x3B1;&#x22C5;(&#x2212;X&#x1D40;(Y&#x2212;P)/m)  |  linear convergence  |  needs lr tuning</text>
  <text x="175" y="295" text-anchor="middle" font-size="9" fill="#555">IRLS (Newton): &#x3B8;&#x2C7C; &#x2190; &#x3B8;&#x2C7C;+(X&#x1D40;W&#x2C7C;X)&#x207B;&#xB9;X&#x1D40;(y&#x2C7C;&#x2212;&#x3BC;&#x2C7C;)  |  quadratic convergence  |  no lr</text>
</svg>
"""

display(SVG(svg))

## Model / Hypothesis

**Categorical distribution in exponential family form** (with $K-1$ free parameters; class $K$ is the reference):

$\displaystyle p(y \mid \eta) = \exp\!\left(\sum_{k=1}^{K-1} \eta_k \,\mathbf{1}[y=k] - a(\eta)\right)$

where the **natural parameters**, **sufficient statistic**, and **log-partition** are:

$\displaystyle \eta_k = \log\frac{\theta_k}{\theta_K} \;\;(\text{log-odds vs reference}), \quad T(y) = (\mathbf{1}[y=1], \ldots, \mathbf{1}[y=K{-}1])^T$

$\displaystyle a(\eta) = \log\!\left(1 + \sum_{k=1}^{K-1} e^{\eta_k}\right) \quad \text{(normalisation constant)}$

**GLM choice**: set $\eta_k = \theta_k^T x$ (linear predictor per class). This connects features to natural parameters directly.

**Canonical response function** (mean as a function of $\eta$, via $\mu_k = \partial a / \partial \eta_k$):

$\displaystyle \mu_k = P(y=k \mid x) = \frac{e^{\theta_k^T x}}{\sum_{j=1}^{K} e^{\theta_j^T x}} \quad \text{(softmax, with } \theta_K = 0 \text{ as reference)}$

The softmax is not *assumed* — it is *derived* as the canonical inverse link for the categorical distribution.

## Derivation

**Step 1 — Categorical → Exponential Family**

Starting from $p(y \mid \theta) = \prod_{k=1}^{K} \theta_k^{\mathbf{1}[y=k]}$, take logs and substitute $\theta_k = e^{\eta_k} / (1 + \sum_j e^{\eta_j})$:

$\displaystyle \log p = \sum_{k=1}^{K-1} \eta_k \,\mathbf{1}[y=k] - \log\!\left(1 + \textstyle\sum_{j=1}^{K-1} e^{\eta_j}\right)$

This matches the exponential family template $\eta^T T(y) - a(\eta)$. ✓

---

**Step 2 — Canonical Response (softmax)**

The mean parameter is $\mu_k = E[\mathbf{1}[y=k]] = P(y=k)$. For exponential families this equals $\partial a / \partial \eta_k$:

$\displaystyle \mu_k = \frac{\partial a}{\partial \eta_k} = \frac{e^{\eta_k}}{1 + \sum_{j=1}^{K-1} e^{\eta_j}}$

Setting the reference parameter $\eta_K = 0$ and including class $K$:

$\displaystyle \boxed{\mu_k = \frac{e^{\eta_k}}{\sum_{j=1}^{K} e^{\eta_j}}} \quad \longleftarrow \text{softmax}$

---

**Step 3 — Universal Gradient (canonical link property)**

For any GLM with canonical link, the gradient of the log-likelihood w.r.t. $\theta_k$ simplifies to:

$\displaystyle \nabla_{\theta_k} \ell = \sum_{i=1}^{m} (y_k^{(i)} - \mu_k^{(i)}) x^{(i)} = X^T(y_k - \mu_k)$

where $y_k \in \{0,1\}^m$ is the binary indicator for class $k$. The same form holds for all $k$ — the coupling between classes enters only through $\mu_k$ (which depends on all $\theta_j$ via softmax).

---

**Step 4 — Hessian and IRLS Update**

The diagonal block of the Hessian for class $k$ (treating cross-class curvature as negligible for the block-diagonal approximation):

$\displaystyle H_{kk} = -X^T W_k X, \qquad W_k = \mathrm{diag}(\mu_k^{(1)}(1-\mu_k^{(1)}),\, \ldots,\, \mu_k^{(m)}(1-\mu_k^{(m)}))$

IRLS Newton step per class $k$:

$\displaystyle \boxed{\theta_k^{(t+1)} = \theta_k^{(t)} + (X^T W_k X)^{-1} X^T(y_k - \mu_k)}$

This treats each class as an independent logistic regression — a **block-diagonal IRLS** approximation. It converges faster than gradient descent but is not the full multinomial Newton step.

## Algorithm: Block-Diagonal IRLS for Softmax GLM

Treats each of the $K$ classes as an independent logistic regression sub-problem, using the current softmax probabilities for all classes to compute each class's working weights.

**Step 1 — Initialise**

Set $\Theta \leftarrow \mathbf{0} \in \mathbb{R}^{(d+1) \times K}$. Prepend bias to $X$. One-hot encode $y$ into $Y$.

**Step 2 — Compute softmax probabilities (all classes jointly)**

$\displaystyle \eta_k = \theta_k^T x^{(i)}, \quad Z = X\Theta \in \mathbb{R}^{m \times K}, \quad P = \text{softmax}(Z)$

**Step 3 — Per-class IRLS Newton step**

For each class $k = 1, \ldots, K$:

$\quad$ a) $\mu_k = P[:, k] \in \mathbb{R}^m$ (softmax probability for class $k$)

$\quad$ b) $w_k = \mu_k (1 - \mu_k)$ (IRLS weights, variance of Bernoulli with $\mu_k$)

$\quad$ c) $\Delta\theta_k = (X^T W_k X)^{-1} X^T(y_k - \mu_k)$ via `np.linalg.solve`

$\quad$ d) $\theta_k \leftarrow \theta_k + \Delta\theta_k$

**Step 4 — Repeat** steps 2–3 until $\|X^T(Y - P)\|_F < \varepsilon$ or maximum iterations reached.

| Quantity | Formula | Shape |
|----------|---------|-------|
| Logits | $Z = X\Theta$ | $(m, K)$ |
| Probabilities | $P = \text{softmax}(Z)$ | $(m, K)$ |
| IRLS weight (class $k$) | $w_k = \mu_k(1-\mu_k)$ | $(m)$ |
| Gradient (class $k$) | $g_k = X^T(y_k - \mu_k)$ | $(d+1)$ |
| Hessian (class $k$) | $H_k = -X^T W_k X$ | $(d+1, d+1)$ |
| IRLS step (class $k$) | $\Delta\theta_k = -H_k^{-1} g_k$ | $(d+1)$ |

## Key Properties

**Softmax is derived, not assumed** — the softmax function is the canonical response function of the categorical exponential family. It emerges from $\mu_k = \partial a / \partial \eta_k$; using any other response function would violate the canonical GLM property and lose the universal gradient form.

**Identifiability** — the model has $K \cdot (d+1)$ parameters but only $K{-}1$ are independently identifiable (adding any vector to all columns of $\Theta$ leaves probabilities unchanged). Fix by setting $\theta_K = 0$ or applying L2 regularisation.

**Block-diagonal vs full Newton** — the full multinomial Hessian is $(K(d+1)) \times (K(d+1))$ with off-diagonal blocks $H_{jk} = -X^T \text{diag}(\mu_j \mu_k) X$ for $j \neq k$. The block-diagonal IRLS ignores these cross terms and updates each class independently. This converges faster than gradient descent but slower than the full Newton step.

**Reduction to logistic regression** — for $K=2$, fixing $\theta_2 = 0$, the block-diagonal IRLS for class 1 becomes exactly the IRLS for logistic regression:

$\displaystyle \theta_1 \leftarrow \theta_1 + (X^T W X)^{-1} X^T(y - \mu_1), \quad W = \text{diag}(\mu_1(1-\mu_1))$

**Convergence** — the categorical log-likelihood is concave in $\Theta$ for canonical link. Block-diagonal IRLS converges monotonically (each step increases the log-likelihood). Full Newton step has quadratic convergence near the optimum.

In [ ]:
import numpy as np


def softmax_stable(Z):
    """
    Numerically stable softmax using the max-shift trick.

    Inputs
    ------
    Z : np.ndarray, shape (m, K)  — logit matrix (Z = X @ Theta)

    Output
    ------
    P : np.ndarray, shape (m, K)  — probability matrix; rows sum to 1
    """
    Z_shifted = Z - Z.max(axis=1, keepdims=True)
    exp_Z     = np.exp(Z_shifted)
    return exp_Z / exp_Z.sum(axis=1, keepdims=True)


def one_hot(y, K):
    """
    Convert integer class labels to one-hot encoding.

    Inputs
    ------
    y : np.ndarray, shape (m,)  — integer labels in {0, ..., K-1}
    K : int                     — number of classes

    Output
    ------
    Y : np.ndarray, shape (m, K)  — one-hot matrix; Y[i, y[i]] = 1
    """
    Y = np.zeros((len(y), K))
    Y[np.arange(len(y)), y] = 1.0
    return Y


def fit_softmax_glm(X, y, K, max_iter=30, tol=1e-8):
    """
    Fit softmax regression via block-diagonal IRLS (Newton-Raphson per class).
    Each class k is updated as an independent logistic regression step,
    using the current softmax probabilities from the full Theta matrix.

    This is the GLM / exponential family approach:
      natural parameter: eta_k = theta_k^T x (canonical link)
      canonical response: mu_k = softmax(eta)_k
      IRLS weight: w_k = mu_k * (1 - mu_k)   [Bernoulli variance for class k]
      Newton step per class k: theta_k += (X^T W_k X)^-1 X^T (y_k - mu_k)

    Inputs
    ------
    X        : np.ndarray, shape (m, d+1)  — design matrix with bias column (x_0 = 1) prepended
    y        : np.ndarray, shape (m,)      — integer class labels in {0, ..., K-1}
    K        : int                         — number of classes
    max_iter : int                         — maximum IRLS iterations
    tol      : float                       — stop when ||X^T(Y-P)||_F < tol

    Outputs
    -------
    Theta   : np.ndarray, shape (d+1, K)  — learned parameter matrix (col k = theta_k)
    history : list of (int, float)         — (iteration, cross-entropy loss) at each step
    """
    m, d    = X.shape
    Theta   = np.zeros((d, K))
    Y       = one_hot(y, K)              # (m, K) one-hot labels
    history = []

    for t in range(max_iter):
        Z  = X @ Theta                   # (m, K)  logits: eta_k = theta_k^T x
        P  = softmax_stable(Z)           # (m, K)  mu_k = canonical response

        # Per-class IRLS Newton step
        for k in range(K):
            mu_k = P[:, k]               # (m,)  softmax probability for class k
            w_k  = mu_k * (1.0 - mu_k)  # (m,)  Bernoulli variance V(mu_k)
            g_k  = X.T @ (Y[:, k] - mu_k)       # (d+1,) gradient = X^T(y_k - mu_k)
            H_k  = -(X.T * w_k) @ X             # (d+1, d+1) Hessian = -X^T W_k X
            Theta[:, k] += np.linalg.solve(-H_k, g_k)   # Newton step

        # Cross-entropy loss
        Z2   = X @ Theta
        P2   = softmax_stable(Z2)
        lse  = np.log(np.exp(Z2 - Z2.max(axis=1, keepdims=True)).sum(axis=1)) + Z2.max(axis=1)
        loss = float(np.mean(lse - Z2[np.arange(m), y]))
        history.append((t, loss))

        # Convergence check
        grad_norm = np.linalg.norm(X.T @ (Y - P2))
        if grad_norm < tol:
            break

    return Theta, history


def predict_proba(X, Theta):
    """
    Return class probability estimates using canonical response (softmax).

    Inputs
    ------
    X     : np.ndarray, shape (m, d+1)    — design matrix with bias column (x_0 = 1) prepended
    Theta : np.ndarray, shape (d+1, K)   — learned parameter matrix

    Output
    ------
    P : np.ndarray, shape (m, K)  — P[i, k] = P(y=k | x^(i))
    """
    return softmax_stable(X @ Theta)


def predict(X, Theta):
    """
    Return predicted class labels (argmax of canonical response).

    Inputs
    ------
    X     : np.ndarray, shape (m, d+1)    — design matrix with bias column (x_0 = 1) prepended
    Theta : np.ndarray, shape (d+1, K)   — learned parameter matrix

    Output
    ------
    y_hat : np.ndarray, shape (m,)  — predicted integer class labels in {0, ..., K-1}
    """
    return predict_proba(X, Theta).argmax(axis=1)